# Create a photon list for a single pointing

In order to create the photon list, I interpolate spectra in temperature and abundance. Please refer to the corresponding notebook for the creation of the table of spectra used for interpolation

## Imports

In [1]:
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from astropy.io import fits
import jax.numpy as jnp
import numpy as np
import cmasher as cmr
import jax.random as random

from xifu_cluster_sim.xifu_config import XIFU_Config
from xifu_cluster_sim.cluster_cube import ClusterCube
from xifu_cluster_sim.photon_list import PhotonList, create_photon_list

## Load (or create) cluster object

### Loading 

In [2]:
cluster = ClusterCube()
cluster.load_cube('path_to_cube.npy')

FileNotFoundError: [Errno 2] No such file or directory: 'path_to_cube.npy'

### Or creating

In [7]:
cluster = ClusterCube()
cluster.create_density_cube()
cluster.create_norm_cube()
cluster.create_kT_cube()
cluster.create_Z_cube()
rng_key = random.PRNGKey(42)
cluster.create_velocity_cube(rng_key)
cluster.convert_velocity_to_redshift()

## Create photon list

Specify the energy grid before, this grid shoulf be the same as the one used for the interpolated table of spectra. X-IFU range of energy is 0.2 to 12 keV. The energy step needs to be much smaller than the energy resolution.

In [8]:

E_low = 0.2
E_high = 12.
E_step = 4e-4
E_n_steps = int((E_high-E_low)/E_step + 1)
E_bounds = np.linspace(E_low,
                       E_high, 
                       E_n_steps) 

Load the table of spectra for interpolation

In [9]:
data = np.load('../../data/spectres_interpoles_T_Z.npz')

T_scale = data['T_scale']
Z_scale = data['Z_scale']
spectra = data['spectra']

Initialize the photon list class

In [10]:
ph_list = PhotonList(cluster,
                    E_bounds,
                    Z_scale,
                    T_scale, 
                    spectra)

Choose a pointing (the cluster cube can be larger than a single X-IFU pointing, to allow for several pointings to be simulated. 

In [11]:
ph_list.select_sub_pointing(x_offset=0,
                            y_offset=0,
                            pointing_shape = (58,58)
                           )

Create the photon list. For memory management purposes, the actual function to call for that is outside the `PhotonList` class. The architecture used (multiprocessor + multithreaded) is not compatible with progress bars unfortunately. It should not take more than a minute in the following example on 32 cores.

In [12]:
# Call the arguments for parallelization 
#(this is essentially a division of the list of spectral properties to use for the model on several processors)
args = ph_list.get_args_list(numprocs = 8, 
                             numthreads = 4)
ph_list = create_photon_list(ph_list,
                            args, 
                            numproc = 8)

/home/mola/miniconda3/envs/myenv/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/home/mola/miniconda3/envs/myenv/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Write the photon list to simputfiles that can be used by SIXTE. Since I use multiprocessing throughout all this project, I divide the photon list into N_procs parts.

In [14]:
ph_list.write_photon_list_to_simputs(results,
                                     num_divisions=32,
                                     path = sim_path,
                                     name_format = 'ph_list')

AttributeError: 'numpy.ndarray' object has no attribute 'write_photon_list_to_simputs'